In [2]:
import pandas as pd
import numpy as np
from sklearn.neighbors import KNeighborsClassifier
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, log_loss, roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer, make_column_selector
import os
os.chdir('/home/pgcp-ai/MachineLearning/Datasets/IrrigationNeed/')

In [4]:
irrigation = pd.read_csv("train.csv", index_col = 0)
irrigation

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
id,,,,,,,,,,,,,,,,,,,,
0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,16.79,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,3.39,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,3.85,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,2.31,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,13.94,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
629995,Clay,6.54,13.45,1.15,1.86,26.65,26.86,1041.33,10.62,18.85,Rice,Sowing,Kharif,Sprinkler,River,4.35,No,118.36,South,Medium
629996,Clay,7.03,54.49,0.96,2.35,36.99,88.00,1419.57,9.93,17.99,Sugarcane,Vegetative,Kharif,Drip,Groundwater,12.97,Yes,40.75,Central,Medium
629997,Clay,6.52,11.98,0.93,0.38,37.82,70.98,88.45,8.19,17.25,Potato,Vegetative,Zaid,Canal,Reservoir,13.58,Yes,2.62,South,High


In [5]:
irrigation.info()

<class 'pandas.core.frame.DataFrame'>
Index: 630000 entries, 0 to 629999
Data columns (total 20 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   Soil_Type                630000 non-null  object 
 1   Soil_pH                  630000 non-null  float64
 2   Soil_Moisture            630000 non-null  float64
 3   Organic_Carbon           630000 non-null  float64
 4   Electrical_Conductivity  630000 non-null  float64
 5   Temperature_C            630000 non-null  float64
 6   Humidity                 630000 non-null  float64
 7   Rainfall_mm              630000 non-null  float64
 8   Sunlight_Hours           630000 non-null  float64
 9   Wind_Speed_kmh           630000 non-null  float64
 10  Crop_Type                630000 non-null  object 
 11  Crop_Growth_Stage        630000 non-null  object 
 12  Season                   630000 non-null  object 
 13  Irrigation_Type          630000 non-null  object 
 14  Water_Sou

In [6]:
irrigation.isna().sum().sum()

0

In [8]:
X, y = irrigation.drop("Irrigation_Need", axis = 1), irrigation["Irrigation_Need"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, random_state = 26,stratify=irrigation['Irrigation_Need'])


In [9]:
ohe = OneHotEncoder(sparse_output = False, drop = 'first').set_output(transform = 'pandas')
transformer = ColumnTransformer(transformers=[('OHE',ohe,make_column_selector(dtype_include=object)),
                                            ]
                               ,remainder='passthrough',
                               verbose_feature_names_out=False
                               ).set_output(transform='pandas')

In [10]:
X_train_ohe = transformer.fit_transform(X_train)
X_test_ohe = transformer.transform(X_test)

In [11]:
ss = StandardScaler()

In [12]:
X_train_scaled = ss.fit_transform(X_train_ohe)
X_test_scaled = ss.transform(X_test_ohe)

In [13]:
scores=[]
k=[1,2,3,4,5,6,7,8,9,10]
for i in tqdm(k):
    knn = KNeighborsClassifier(n_neighbors=i)
    knn.fit(X_train_scaled,y_train)
    y_pred = knn.predict_proba(X_test_scaled)
    score = log_loss(y_test,y_pred)
    scores.append([i,score])
df_scores = pd.DataFrame(scores,columns=['k','scores'])
df_scores.sort_values('scores',ascending=True)

100%|██████████████████████████████████████████| 10/10 [23:55<00:00, 143.54s/it]


,k,scores
9,10,0.613392
8,9,0.654846
7,8,0.715651
6,7,0.805283
5,6,0.936940
4,5,1.146693
3,4,1.518504
2,3,2.247967
1,2,3.890114
0,1,8.711122


In [15]:
X_ohe = transformer.fit_transform(X)
X_scaled = ss.fit_transform(X_ohe)
bm = KNeighborsClassifier(n_neighbors = 10, n_jobs = -1)
bm.fit(X_scaled, y)

KNeighborsClassifier(n_jobs=-1, n_neighbors=10)

In [16]:
tst = pd.read_csv("test.csv",index_col='id')
tst_ohe = transformer.transform(tst)
tst_scaled = ss.transform(tst_ohe)
y_pred_proba = bm.predict_proba(tst_scaled)

In [17]:
sample = pd.read_csv("sample_submission.csv")
sample["Irrigation_Need"] = bm.predict(tst_scaled)
sample

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low
...,...,...
269995,899995,Low
269996,899996,Low
269997,899997,Medium
269998,899998,Low


In [18]:
sample["Irrigation_Need"].value_counts()

Irrigation_Need
Low       173255
Medium     94867
High        1878
Name: count, dtype: int64

In [19]:
sample.to_csv("KaggleSubmissionUsingKNearesNeighbors.csv", index = False)